<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/RNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Example image](https://upload.wikimedia.org/wikipedia/commons/0/02/Northeastern_Wordmark.svg)

# Recurrent Neural Network (RNN) Example

Copyright: Prof. Shanu Sushmita

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.models import Sequential

In [ ]:
# Define the probability distribution and vocabulary
probabilities = [0.13, 0.00, 0.84, 0.03]
vocabulary = ['h', 'e', 'l', 'o']

# Ensure the probabilities sum to 1
assert sum(probabilities) == 1, "Probabilities must sum to 1."

# Generate a sample character based on the given probabilities
sample_character = np.random.choice(vocabulary, p=probabilities)

print(f"Sampled character: {sample_character}")

Sampled character: h


In [ ]:
# Sample text for training
text = """
This is a sample text for character generation using recurrent neural networks.
We will train the network to predict the next character given a sequence of characters.
"""

In [ ]:
# Creating character mappings
chars = sorted(list(set(text)))
char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}
num_chars = len(chars)

In [ ]:
# Create training data
seq_length = 40
X_data = []
y_data = []
for i in range(len(text) - seq_length):
    seq_in = text[i:i+seq_length]
    seq_out = text[i+seq_length]
    X_data.append([char_to_idx[char] for char in seq_in])
    y_data.append(char_to_idx[seq_out])

In [ ]:
# Reshape and normalize data
X = np.reshape(X_data, (len(X_data), seq_length, 1))
X = X / float(num_chars)
y = tf.keras.utils.to_categorical(y_data)

In [ ]:
# Build the model
model = Sequential([
    LSTM(128, input_shape=(X.shape[1], X.shape[2])),
    Dense(num_chars, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
# Train the model
model.fit(X, y, epochs=20, batch_size=64)

In [ ]:
# Function to generate text
def generate_text(model, start_seed, num_chars_to_generate=100):
    generated_text = start_seed
    for i in range(num_chars_to_generate):
        x_pred = np.reshape([char_to_idx[char] for char in generated_text[-seq_length:]], (1, seq_length, 1))
        x_pred = x_pred / float(num_chars)
        predicted_probs = model.predict(x_pred, verbose=0)[0]
        predicted_char_idx = np.random.choice(len(chars), p=predicted_probs)
        predicted_char = idx_to_char[predicted_char_idx]
        generated_text += predicted_char
    return generated_text

In [ ]:
# Generate text
start_seed = "This is a sample text for character gene"
generated_text = generate_text(model, start_seed, num_chars_to_generate=200)
print(generated_text)

## IMDB Movie Dataset

In [ ]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Embedding, Dropout
from tensorflow.keras.preprocessing import sequence

In [ ]:
# Load IMDB dataset
max_features = 20000
maxlen = 80  # cut texts after this number of words (among top max_features most common words)
batch_size = 32

In [ ]:
print('Loading data...')
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')
print('Pad sequences (samples x time)')
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)

In [ ]:
## Complete next steps for the model building